In [1]:
import numpy as np
import pandas as pd
import torch
import catboost

In [2]:
df_interactions = pd.read_csv("data_original/interactions.csv")

In [3]:
df_items = pd.read_csv("data_original/items.csv")

In [4]:
df_users = pd.read_csv("data_original/users.csv")

In [5]:
df_users

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0
...,...,...,...,...,...
840192,339025,age_65_inf,income_0_20,Ж,0
840193,983617,age_18_24,income_20_40,Ж,1
840194,251008,NaN,NaN,NaN,0
840195,590706,NaN,NaN,Ж,0


In [6]:
df_items

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15958,6443,series,Полярный круг,Arctic Circle,2018.0,"драмы, триллеры, криминал","Финляндия, Германия",NaN,16.0,NaN,Ханну Салонен,"Иина Куустонен, Максимилиан Брюкнер, Пихла Вии...","Во время погони за браконьерами по лесу, сотру...","убийство, вирус, расследование преступления, н..."
15959,2367,series,Надежда,NaN,2020.0,"драмы, боевики",Россия,0.0,18.0,NaN,Елена Хазанова,"Виктория Исакова, Александр Кузьмин, Алексей М...",Оригинальный киносериал от создателей «Бывших»...,"Надежда, 2020, Россия"
15960,10632,series,Сговор,Hassel,2017.0,"драмы, триллеры, криминал",Россия,0.0,18.0,NaN,"Эшреф Рейбрук, Амир Камдин, Эрик Эгер","Ола Рапас, Алиетт Офейм, Уильма Лиден, Шанти Р...",Криминальная драма по мотивам романов о шведск...,"Сговор, 2017, Россия"
15961,4538,series,Среди камней,Darklands,2019.0,"драмы, спорт, криминал",Россия,0.0,18.0,NaN,"Марк О’Коннор, Конор МакМахон","Дэйн Уайт О’Хара, Томас Кэйн-Бирн, Джудит Родд...",Семнадцатилетний Дэмиен мечтает вырваться за п...,"Среди, камней, 2019, Россия"


In [7]:
df_interactions

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250,72.0
1,699317,1659,2021-05-29,8317,100.0
2,656683,7107,2021-05-09,10,0.0
3,864613,7638,2021-07-05,14483,100.0
4,964868,9506,2021-04-30,6725,100.0
...,...,...,...,...,...
5476246,648596,12225,2021-08-13,76,0.0
5476247,546862,9673,2021-04-13,2308,49.0
5476248,697262,15297,2021-08-20,18307,63.0
5476249,384202,16197,2021-04-19,6203,100.0


In [8]:
df_interactions["last_watch_dt"] = pd.to_datetime(
    df_interactions["last_watch_dt"]
)

print("Начало:", df_interactions["last_watch_dt"].min())
print("Конец:", df_interactions["last_watch_dt"].max())

print("\nВзаимодействий на пользователя:")
print(
    df_interactions.groupby("user_id")
    .size()
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95])
)

print("\nПроцент просмотра:")
print(df_interactions["watched_pct"].describe())

print("\nПовторяющиеся user-item:")
print(
    df_interactions.duplicated(["user_id", "item_id"]).sum()
)

Начало: 2021-03-13 00:00:00
Конец: 2021-08-22 00:00:00

Взаимодействий на пользователя:
count    962179.000000
mean          5.691510
std          10.886262
min           1.000000
50%           2.000000
75%           6.000000
90%          13.000000
95%          21.000000
max        1341.000000
dtype: float64

Процент просмотра:
count    5.475423e+06
mean     4.637400e+01
std      4.191853e+01
min      0.000000e+00
25%      4.000000e+00
50%      3.200000e+01
75%      1.000000e+02
max      1.000000e+02
Name: watched_pct, dtype: float64

Повторяющиеся user-item:
0


In [9]:
weekly = (
    df_interactions
    .set_index("last_watch_dt")
    .resample("W-SUN")
    .agg(
        interactions=("user_id", "size"),
        users=("user_id", "nunique")
    )
)

print(weekly.tail(12))

               interactions   users
last_watch_dt                      
2021-06-06           214709   97788
2021-06-13           253048  116301
2021-06-20           307934  141928
2021-06-27           295131  126038
2021-07-04           357123  160798
2021-07-11           299513  129511
2021-07-18           321120  133961
2021-07-25           332313  135221
2021-08-01           373455  150593
2021-08-08           383149  152466
2021-08-15           402653  160794
2021-08-22           424436  167348


In [10]:
cutoff = pd.Timestamp("2021-07-19")

history = df_interactions[
    df_interactions["last_watch_dt"] < cutoff
]

user_counts = history.groupby("user_id").size()

eligible_users = user_counts[
    user_counts >= 3
].index

sampled_users = pd.Series(eligible_users).sample(
    n=min(10000, len(eligible_users)),
    random_state=42
)

df_small = df_interactions[
    df_interactions["user_id"].isin(sampled_users)
].copy()

print(df_small.shape)
print(df_small["user_id"].nunique())

(120222, 5)
10000


In [11]:
periods = {
    "train_labels": ("2021-07-19", "2021-08-09"),
    "validation": ("2021-08-09", "2021-08-16"),
    "test": ("2021-08-16", "2021-08-23")
}

for name, (start, end) in periods.items():
    part = df_small[
        (df_small["last_watch_dt"] >= start) &
        (df_small["last_watch_dt"] < end)
    ]

    print(name, len(part), part["user_id"].nunique())

train_labels 15875 3545
validation 4494 1643
test 4302 1548


In [12]:
import numpy as np
import pandas as pd

def evaluate_popular(df, cutoff, k=200):
    cutoff = pd.Timestamp(cutoff)

    history = df[df["last_watch_dt"] < cutoff]

    future = df[
        (df["last_watch_dt"] >= cutoff) &
        (df["last_watch_dt"] < cutoff + pd.Timedelta(days=7)) &
        (df["watched_pct"] >= 50)
    ]

    popular = history["item_id"].value_counts().head(1000).index.tolist()

    seen = history.groupby("user_id")["item_id"].agg(set).to_dict()
    targets = future.groupby("user_id")["item_id"].agg(set).to_dict()

    recalls = []

    for user, target_items in targets.items():
        if user not in seen:
            continue

        relevant = target_items - seen[user]

        if not relevant:
            continue

        candidates = [
            item for item in popular
            if item not in seen[user]
        ][:k]

        recall = len(set(candidates) & relevant) / len(relevant)
        recalls.append(recall)

    print("Пользователей в оценке:", len(recalls))
    print(f"Recall@{k}:", np.mean(recalls) if recalls else float("nan"))

In [13]:
evaluate_popular(df_small, "2021-08-09", k=200)

Пользователей в оценке: 876
Recall@200: 0.3871738089707469


In [14]:
def build_pairs(df, cutoff, k=200, training=False):
    cutoff = pd.Timestamp(cutoff)

    history = df[df["last_watch_dt"] < cutoff]
    future = df[
        (df["last_watch_dt"] >= cutoff) &
        (df["last_watch_dt"] < cutoff + pd.Timedelta(days=7)) &
        (df["watched_pct"] >= 50)
    ]

    popular = history["item_id"].value_counts().head(1000).index.tolist()

    seen = history.groupby("user_id")["item_id"].agg(set).to_dict()
    targets = future.groupby("user_id")["item_id"].agg(set).to_dict()

    rng = np.random.default_rng(42)
    rows = []

    for user, truth in targets.items():
        if user not in seen:
            continue

        truth = truth - seen[user]
        if not truth:
            continue

        candidates = [
            item for item in popular
            if item not in seen[user]
        ][:k]

        if training:
            positives = [i for i in candidates if i in truth]
            if not positives:
                continue

            negatives = [i for i in candidates if i not in truth]
            n_neg = min(len(negatives), 20 * len(positives))

            negatives = rng.choice(
                negatives, size=n_neg, replace=False
            ).tolist()

            candidates = positives + negatives

        rows.extend(
            (user, item, int(item in truth), cutoff)
            for item in candidates
        )

    return pd.DataFrame(
        rows,
        columns=["user_id", "item_id", "label", "cutoff"]
    )

In [15]:
train_cutoffs = [
    "2021-07-19",
    "2021-07-26",
    "2021-08-02"
]

df_train = pd.concat(
    [build_pairs(df_small, t, training=True) for t in train_cutoffs],
    ignore_index=True
)

df_val = build_pairs(
    df_small, "2021-08-09", training=False
)

print("Train:", df_train.shape)
print("Validation:", df_val.shape)
print("Train labels:", df_train["label"].value_counts())

Train: (50422, 4)
Validation: (175200, 4)
Train labels: label
0    48018
1     2404
Name: count, dtype: int64


In [16]:
def add_features(pairs):
    result = []

    item_meta = df_items[
        ["item_id", "release_year", "age_rating", "content_type", "genres"]
    ].copy()

    item_meta["main_genre"] = (
        item_meta["genres"].fillna("unknown")
        .str.split(",").str[0].str.strip()
    )

    for cutoff, part in pairs.groupby("cutoff"):
        history = df_small[df_small["last_watch_dt"] < cutoff]
        f = part.copy()

        for days in [7, 30]:
            h = history[
                history["last_watch_dt"] >= cutoff - pd.Timedelta(days=days)
            ]

            for key, prefix in [
                ("user_id", "user"),
                ("item_id", "item")
            ]:
                counts = h.groupby(key).size()
                f[f"{prefix}_count_{days}d"] = (
                    f[key].map(counts).fillna(0)
                )

        last_watch = history.groupby("user_id")["last_watch_dt"].max()
        f["user_recency"] = (
            cutoff - f["user_id"].map(last_watch)
        ).dt.days

        f = f.merge(
            item_meta.drop(columns="genres"),
            on="item_id",
            how="left"
        )

        h30 = history[
            history["last_watch_dt"] >= cutoff - pd.Timedelta(days=30)
        ].merge(
            item_meta[["item_id", "main_genre"]],
            on="item_id",
            how="left"
        )

        genre_counts = (
            h30.groupby(["user_id", "main_genre"])
            .size()
            .rename("user_genre_count_30d")
            .reset_index()
        )

        f = f.merge(
            genre_counts,
            on=["user_id", "main_genre"],
            how="left"
        )

        f["user_genre_count_30d"] = (
            f["user_genre_count_30d"].fillna(0)
        )

        f["user_genre_share_30d"] = (
            f["user_genre_count_30d"] /
            (f["user_count_30d"] + 1)
        )

        f = f.merge(
            df_users,
            on="user_id",
            how="left"
        )

        for c in ["content_type", "main_genre", "age", "income", "sex"]:
            f[c] = f[c].fillna("unknown").astype(str)

        result.append(f)

    return pd.concat(result, ignore_index=True)

In [17]:
train_features = add_features(df_train)
val_features = add_features(df_val)

print(train_features.shape)
print(val_features.shape)

(50422, 19)
(175200, 19)


In [18]:
from catboost import CatBoostClassifier

features = [
    "user_count_7d", "user_count_30d",
    "item_count_7d", "item_count_30d",
    "user_recency",
    "user_genre_count_30d",
    "user_genre_share_30d",
    "release_year", "age_rating",
    "content_type", "main_genre",
    "age", "income", "sex", "kids_flg"
]

cat_features = [
    "content_type", "main_genre",
    "age", "income", "sex"
]

model = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=42,
    verbose=100
)

model.fit(
    train_features[features],
    train_features["label"],
    cat_features=cat_features,
    eval_set=(
        val_features[features],
        val_features["label"]
    ),
    early_stopping_rounds=60
)

val_features["score"] = model.predict_proba(
    val_features[features]
)[:, 1]

0:	learn: 0.6199553	test: 0.6128181	best: 0.6128181 (0)	total: 418ms	remaining: 4m 10s
100:	learn: 0.1465388	test: 0.0415885	best: 0.0415885 (100)	total: 15.4s	remaining: 1m 16s
200:	learn: 0.1432890	test: 0.0409045	best: 0.0408916 (198)	total: 28s	remaining: 55.6s
300:	learn: 0.1399542	test: 0.0404792	best: 0.0404680 (299)	total: 41.6s	remaining: 41.3s
Stopped by overfitting detector  (60 iterations wait)

bestTest = 0.04046799345
bestIteration = 299

Shrink model to first 300 iterations.


In [19]:
v = val_features.sort_values(
    ["user_id", "score"],
    ascending=[True, False]
).copy()

v["rank"] = v.groupby("user_id").cumcount() + 1

v["dcg_part"] = np.where(
    v["rank"] <= 10,
    v["label"] / np.log2(v["rank"] + 1),
    0
)

dcg = v.groupby("user_id")["dcg_part"].sum()
positive_counts = v.groupby("user_id")["label"].sum().clip(upper=10)

discounts = 1 / np.log2(np.arange(2, 12))

idcg = positive_counts.map(
    lambda n: discounts[:int(n)].sum()
)

ndcg = (dcg / idcg.replace(0, np.nan)).fillna(0).mean()

print("NDCG@10:", ndcg)

NDCG@10: 0.14262804458549544


In [20]:
v = val_features.sort_values(
    ["user_id", "score"],
    ascending=[True, False]
).copy()

v["rank"] = v.groupby("user_id").cumcount() + 1

v["dcg_part"] = np.where(
    v["rank"] <= 10,
    v["label"] / np.log2(v["rank"] + 1),
    0
)

dcg = v.groupby("user_id")["dcg_part"].sum()
positive_counts = v.groupby("user_id")["label"].sum().clip(upper=10)

discounts = 1 / np.log2(np.arange(2, 12))

idcg = positive_counts.map(
    lambda n: discounts[:int(n)].sum()
)

ndcg = (dcg / idcg.replace(0, np.nan)).fillna(0).mean()

print("NDCG@10:", ndcg)

NDCG@10: 0.14262804458549544


In [21]:
cutoff = pd.Timestamp("2021-08-09")

history = df_small[df_small["last_watch_dt"] < cutoff]

future = df_small[
    (df_small["last_watch_dt"] >= cutoff) &
    (df_small["last_watch_dt"] < cutoff + pd.Timedelta(days=7)) &
    (df_small["watched_pct"] >= 50)
]

seen = history.groupby("user_id")["item_id"].agg(set).to_dict()
truth = future.groupby("user_id")["item_id"].agg(set).to_dict()

popularity = history["item_id"].value_counts()
val_features["pop_score"] = (
    val_features["item_id"].map(popularity).fillna(0)
)

def ndcg10_full(df, score_col):
    ranked = df.sort_values(
        ["user_id", score_col],
        ascending=[True, False]
    )

    results = []

    for user, group in ranked.groupby("user_id"):
        relevant = truth.get(user, set()) - seen.get(user, set())
        if not relevant:
            continue

        labels = group["label"].to_numpy()[:10]

        dcg = sum(
            label / np.log2(rank + 2)
            for rank, label in enumerate(labels)
        )

        idcg = sum(
            1 / np.log2(rank + 2)
            for rank in range(min(len(relevant), 10))
        )

        results.append(dcg / idcg)

    return np.mean(results)

print("Top Popular:", ndcg10_full(val_features, "pop_score"))
print("CatBoost:", ndcg10_full(val_features, "score"))

Top Popular: 0.10660864287983428
CatBoost: 0.12176266899853366


In [22]:
import time

genre_map = (
    df_items[["item_id", "genres"]]
    .drop_duplicates("item_id")
    .assign(
        main_genre=lambda d: (
            d["genres"].fillna("unknown")
            .str.split(",").str[0].str.strip()
        )
    )
    .set_index("item_id")["main_genre"]
)

def generate_auto_features(pairs, windows=(3, 7, 14, 30)):
    result = []

    for cutoff, part in pairs.groupby("cutoff"):
        h = df_small.loc[
            df_small["last_watch_dt"] < cutoff,
            ["user_id", "item_id", "last_watch_dt", "watched_pct"]
        ].copy()

        h["main_genre"] = h["item_id"].map(genre_map).fillna("unknown")

        f = part.copy()
        if "main_genre" not in f.columns:
            f["main_genre"] = f["item_id"].map(genre_map).fillna("unknown")

        for w in windows:
            hw = h[
                h["last_watch_dt"] >= cutoff - pd.Timedelta(days=w)
            ]

            groups = [
                (["user_id"], "user", f["user_id"]),
                (["item_id"], "item", f["item_id"]),
                (
                    ["user_id", "main_genre"],
                    "user_genre",
                    pd.MultiIndex.from_frame(f[["user_id", "main_genre"]])
                )
            ]

            for keys, name, idx in groups:
                grouped = hw.groupby(keys)["watched_pct"]

                f[f"auto_{name}_count_{w}d"] = (
                    grouped.size()
                    .reindex(idx, fill_value=0)
                    .to_numpy(dtype="float32")
                )

                f[f"auto_{name}_mean_pct_{w}d"] = (
                    grouped.mean()
                    .reindex(idx)
                    .fillna(0)
                    .to_numpy(dtype="float32")
                )

        result.append(f)

    return pd.concat(result, ignore_index=True)

In [23]:
start = time.perf_counter()

train_af = generate_auto_features(train_features)
val_af = generate_auto_features(val_features)

feature_time = time.perf_counter() - start

existing = {
    "auto_user_count_7d",
    "auto_user_count_30d",
    "auto_item_count_7d",
    "auto_item_count_30d",
    "auto_user_genre_count_30d"
}

auto_cols = [
    c for c in train_af.columns
    if c.startswith("auto_") and c not in existing
]

print("Новых признаков:", len(auto_cols))
print("Время генерации:", round(feature_time, 2), "сек")
print("Train:", train_af.shape)
print("Validation:", val_af.shape)

Новых признаков: 19
Время генерации: 0.72 сек
Train: (50422, 43)
Validation: (175200, 45)


In [24]:
all_features = features + auto_cols

model_auto = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=42,
    verbose=100
)

model_auto.fit(
    train_af[all_features],
    train_af["label"],
    cat_features=cat_features,
    eval_set=(val_af[all_features], val_af["label"]),
    early_stopping_rounds=60
)

val_af["score_auto"] = model_auto.predict_proba(
    val_af[all_features]
)[:, 1]

print("Manual CatBoost: 0.1217627")
print("CatBoost + AutoFE:", ndcg10_full(val_af, "score_auto"))

0:	learn: 0.6195038	test: 0.6123551	best: 0.6123551 (0)	total: 166ms	remaining: 1m 39s
100:	learn: 0.1438383	test: 0.0399002	best: 0.0399002 (100)	total: 15.7s	remaining: 1m 17s
Stopped by overfitting detector  (60 iterations wait)

bestTest = 0.03961067248
bestIteration = 126

Shrink model to first 127 iterations.
Manual CatBoost: 0.1217627
CatBoost + AutoFE: 0.1270353425778107


In [25]:
importance = pd.Series(
    model_auto.get_feature_importance(),
    index=all_features
)

ranked = importance[auto_cols].sort_values(ascending=False).index

selected = []

for col in ranked:
    corrs = train_af[selected].corrwith(train_af[col]).abs().fillna(0)

    if not selected or corrs.max() < 0.95:
        selected.append(col)

    if len(selected) == 8:
        break

print("Отобранные признаки:", selected)

Отобранные признаки: ['auto_item_count_14d', 'auto_item_mean_pct_30d', 'auto_item_mean_pct_7d', 'auto_item_count_3d', 'auto_item_mean_pct_14d', 'auto_item_mean_pct_3d', 'auto_user_genre_mean_pct_30d', 'auto_user_mean_pct_3d']


In [26]:
selected_features = features + selected

model_selected = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=42,
    verbose=100
)

model_selected.fit(
    train_af[selected_features],
    train_af["label"],
    cat_features=cat_features,
    eval_set=(
        val_af[selected_features],
        val_af["label"]
    ),
    early_stopping_rounds=60
)

val_af["score_selected"] = model_selected.predict_proba(
    val_af[selected_features]
)[:, 1]

print("Manual:", ndcg10_full(val_af, "score"))
print("All AutoFE:", ndcg10_full(val_af, "score_auto"))
print("Selected AutoFE:", ndcg10_full(val_af, "score_selected"))

0:	learn: 0.6183512	test: 0.6105411	best: 0.6105411 (0)	total: 247ms	remaining: 2m 28s
100:	learn: 0.1442590	test: 0.0398584	best: 0.0398569 (99)	total: 17.5s	remaining: 1m 26s
Stopped by overfitting detector  (60 iterations wait)

bestTest = 0.03955785544
bestIteration = 139

Shrink model to first 140 iterations.
Manual: 0.12176266899853366
All AutoFE: 0.1270353425778107
Selected AutoFE: 0.12495684312183111


In [27]:
def evaluate_ndcg(pairs, score_col, cutoff, k=10):
    cutoff = pd.Timestamp(cutoff)

    history = df_small[df_small["last_watch_dt"] < cutoff]

    future = df_small[
        (df_small["last_watch_dt"] >= cutoff) &
        (df_small["last_watch_dt"] < cutoff + pd.Timedelta(days=7)) &
        (df_small["watched_pct"] >= 50)
    ]

    seen = history.groupby("user_id")["item_id"].agg(set).to_dict()
    truth = future.groupby("user_id")["item_id"].agg(set).to_dict()

    results = []

    for user, group in pairs.groupby("user_id"):
        relevant = truth.get(user, set()) - seen.get(user, set())

        if not relevant:
            continue

        top_items = group.nlargest(k, score_col)["item_id"]

        dcg = sum(
            (item in relevant) / np.log2(rank + 2)
            for rank, item in enumerate(top_items)
        )

        idcg = sum(
            1 / np.log2(rank + 2)
            for rank in range(min(len(relevant), k))
        )

        results.append(dcg / idcg)

    return np.mean(results)

In [28]:
print(evaluate_ndcg(val_af, "score_auto", "2021-08-09"))
print(evaluate_ndcg(val_af, "score_selected", "2021-08-09"))

0.1270353425778107
0.12495684312183111


In [29]:
def build_eval_pairs(df, cutoff, users, k=200):
    cutoff = pd.Timestamp(cutoff)

    history = df[df["last_watch_dt"] < cutoff]

    future = df[
        (df["last_watch_dt"] >= cutoff) &
        (df["last_watch_dt"] < cutoff + pd.Timedelta(days=7)) &
        (df["watched_pct"] >= 50)
    ]

    popular = history["item_id"].value_counts().head(1000).index.tolist()
    seen = history.groupby("user_id")["item_id"].agg(set).to_dict()
    targets = future.groupby("user_id")["item_id"].agg(set).to_dict()

    rows = []

    for user in users:
        if user not in seen:
            continue

        relevant = targets.get(user, set()) - seen[user]

        candidates = [
            item for item in popular
            if item not in seen[user]
        ][:k]

        rows.extend(
            (user, item, int(item in relevant), cutoff)
            for item in candidates
        )

    return pd.DataFrame(
        rows,
        columns=["user_id", "item_id", "label", "cutoff"]
    )

In [30]:
eval_users = (
    df_small.loc[
        df_small["last_watch_dt"] < "2021-07-19",
        "user_id"
    ]
    .drop_duplicates()
    .sample(n=3000, random_state=42)
    .tolist()
)

df_val_fixed = build_eval_pairs(
    df_small, "2021-08-09", eval_users
)

val_fixed = add_features(df_val_fixed)
val_fixed = generate_auto_features(val_fixed)

In [31]:
val_fixed["score_manual"] = model.predict_proba(
    val_fixed[features]
)[:, 1]

val_fixed["score_auto"] = model_auto.predict_proba(
    val_fixed[all_features]
)[:, 1]

val_fixed["score_selected"] = model_selected.predict_proba(
    val_fixed[selected_features]
)[:, 1]

for col in ["score_manual", "score_auto", "score_selected"]:
    print(col, evaluate_ndcg(val_fixed, col, "2021-08-09"))

score_manual 0.11352470160549291
score_auto 0.12600911165103096
score_selected 0.12368124355638632


In [32]:
# 1. Корреляционная матрица
base_num = [c for c in features if c not in cat_features]
cols = base_num + auto_cols

X = train_af[cols].apply(pd.to_numeric, errors="coerce").fillna(0)

X = X - X.groupby([
    train_af["cutoff"],
    train_af["user_id"]
]).transform("mean")

valid = X.std()[lambda s: s > 1e-8].index.tolist()
R = X[valid].corr().fillna(0)

# 2. Нормированная важность
imp = pd.Series(
    model_auto.get_feature_importance(),
    index=all_features
)[auto_cols]

imp = imp / (imp.max() + 1e-12)

# 3. Жадный спектральный отбор
selected_spectral = []
alpha = 0.05

for _ in range(8):
    scores = {}

    for f in auto_cols:
        if f in selected_spectral:
            continue

        if f not in valid:
            novelty = 0.5
        else:
            S = [c for c in base_num + selected_spectral if c in valid]

            r = R.loc[S, f].to_numpy()
            A = R.loc[S, S].to_numpy()

            r2 = r @ np.linalg.solve(
                A + alpha * np.eye(len(S)), r
            )

            novelty = np.clip(1 - r2, 0, 1)

        scores[f] = imp[f] + 0.2 * novelty

    selected_spectral.append(max(scores, key=scores.get))

print("Spectral features:")
print(*selected_spectral, sep="\n")

Spectral features:
auto_item_count_14d
auto_item_mean_pct_30d
auto_item_mean_pct_7d
auto_item_count_3d
auto_item_mean_pct_3d
auto_item_mean_pct_14d
auto_user_mean_pct_3d
auto_user_mean_pct_7d


In [33]:
spectral_features = features + selected_spectral

model_spectral = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    random_seed=42,
    verbose=100
)

model_spectral.fit(
    train_af[spectral_features],
    train_af["label"],
    cat_features=cat_features,
    eval_set=(val_af[spectral_features], val_af["label"]),
    early_stopping_rounds=60
)

val_fixed["score_spectral"] = model_spectral.predict_proba(
    val_fixed[spectral_features]
)[:, 1]

print(
    "Spectral NDCG@10:",
    evaluate_ndcg(val_fixed, "score_spectral", "2021-08-09")
)

0:	learn: 0.6188515	test: 0.6112626	best: 0.6112626 (0)	total: 69.4ms	remaining: 41.6s
100:	learn: 0.1438792	test: 0.0401587	best: 0.0401587 (100)	total: 7.38s	remaining: 36.5s
200:	learn: 0.1397439	test: 0.0399411	best: 0.0398387 (169)	total: 14.4s	remaining: 28.7s
Stopped by overfitting detector  (60 iterations wait)

bestTest = 0.03983866204
bestIteration = 169

Shrink model to first 170 iterations.
Spectral NDCG@10: 0.12137870013842947


In [34]:
geom_pairs = build_eval_pairs(
    df_small,
    "2021-08-02",
    eval_users[:1000],
    k=200
)

geom = add_features(geom_pairs)
geom = generate_auto_features(geom)

base_num = [c for c in features if c not in cat_features]
cols = base_num + auto_cols

X = geom[cols].astype("float32")

# Центрируем внутри каждого пользовательского списка
X = X - X.groupby(
    [geom["cutoff"], geom["user_id"]]
).transform("mean")

valid = X.std()[lambda s: s > 1e-8].index
R = X[valid].corr().fillna(0)

imp = pd.Series(
    model_auto.get_feature_importance(),
    index=all_features
)[auto_cols]

imp = imp / (imp.max() + 1e-12)

In [35]:
selected_v2 = []
alpha = 0.05

for _ in range(8):
    scores = {}

    for f in auto_cols:
        if f in selected_v2:
            continue

        if f not in valid:
            novelty = 0.0
        else:
            S = [
                c for c in base_num + selected_v2
                if c in valid
            ]

            r = R.loc[S, f].to_numpy()
            A = R.loc[S, S].to_numpy()

            novelty = 1 - r @ np.linalg.solve(
                A + alpha * np.eye(len(S)), r
            )
            novelty = np.clip(novelty, 0, 1)

        scores[f] = imp[f] * np.sqrt(0.05 + novelty)

    selected_v2.append(max(scores, key=scores.get))

print(selected_v2)

['auto_item_mean_pct_30d', 'auto_item_mean_pct_7d', 'auto_item_count_14d', 'auto_item_mean_pct_3d', 'auto_item_mean_pct_14d', 'auto_item_count_3d', 'auto_user_genre_mean_pct_30d', 'auto_user_mean_pct_3d']


In [36]:
def add_temporal_features(df):
    df = df.copy()

    c3 = df["auto_item_count_3d"]
    c7 = df["auto_item_count_7d"]
    c14 = df["auto_item_count_14d"]
    c30 = df["auto_item_count_30d"]

    df["item_accel_3_14"] = (
        np.log1p(c3 / 3)
        - np.log1p((c14 - c3).clip(lower=0) / 11)
    )

    df["item_accel_7_30"] = (
        np.log1p(c7 / 7)
        - np.log1p((c30 - c7).clip(lower=0) / 23)
    )

    df["item_quality_shift"] = (
        df["auto_item_mean_pct_7d"]
        - df["auto_item_mean_pct_30d"]
    ) * c7 / (c7 + 5)

    df["genre_interest_shift"] = (
        df["auto_user_genre_count_7d"] /
        (df["auto_user_count_7d"] + 2)
        -
        df["auto_user_genre_count_30d"] /
        (df["auto_user_count_30d"] + 2)
    )

    return df

In [37]:
def fit_variant(name, extra_cols):
    cols = features + extra_cols

    m = CatBoostClassifier(
        iterations=600,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        random_seed=42,
        verbose=False
    )

    m.fit(
        train_af[cols], train_af["label"],
        cat_features=cat_features,
        eval_set=(val_af[cols], val_af["label"]),
        early_stopping_rounds=60
    )

    score_col = "score_" + name

    val_fixed[score_col] = m.predict_proba(
        val_fixed[cols]
    )[:, 1]

    result = evaluate_ndcg(
        val_fixed, score_col, "2021-08-09"
    )

    print(name, "NDCG@10 =", round(result, 6))
    return m

In [38]:
model_spectral_v2 = fit_variant(
    "spectral_v2",
    selected_v2
)

spectral_v2 NDCG@10 = 0.123593


In [39]:
train_af = add_temporal_features(train_af)
val_af = add_temporal_features(val_af)
val_fixed = add_temporal_features(val_fixed)

temporal_cols = [
    "item_accel_3_14",
    "item_accel_7_30",
    "item_quality_shift",
    "genre_interest_shift"
]

model_temporal = fit_variant(
    "temporal",
    auto_cols + temporal_cols
)

temporal NDCG@10 = 0.120302


In [40]:
def add_decay_features(pairs, taus=(3, 14, 30)):
    result = []

    for cutoff, part in pairs.groupby("cutoff"):
        h = df_small[
            df_small["last_watch_dt"] < cutoff
        ].copy()

        h["main_genre"] = (
            h["item_id"].map(genre_map).fillna("unknown")
        )

        f = part.copy()

        if "main_genre" not in f:
            f["main_genre"] = (
                f["item_id"].map(genre_map).fillna("unknown")
            )

        age = (
            (cutoff - h["last_watch_dt"])
            .dt.total_seconds() / 86400
        )

        for tau in taus:
            h["_weight"] = np.exp(-age / tau)

            item_score = h.groupby("item_id")["_weight"].sum()

            f[f"decay_item_{tau}"] = (
                f["item_id"].map(item_score).fillna(0)
            )

            ug_score = (
                h.groupby(["user_id", "main_genre"])["_weight"]
                .sum()
                .rename(f"decay_user_genre_{tau}")
                .reset_index()
            )

            f = f.merge(
                ug_score,
                on=["user_id", "main_genre"],
                how="left"
            )

            col = f"decay_user_genre_{tau}"
            f[col] = f[col].fillna(0)

        result.append(f)

    return pd.concat(result, ignore_index=True)

In [41]:
train_af = add_decay_features(train_af)
val_af = add_decay_features(val_af)
val_fixed = add_decay_features(val_fixed)

decay_cols = [
    f"decay_{entity}_{tau}"
    for entity in ["item", "user_genre"]
    for tau in [3, 14, 30]
]

model_decay = fit_variant(
    "decay",
    auto_cols + decay_cols
)

decay NDCG@10 = 0.121631


In [42]:
model_decay_only = fit_variant(
    "decay_only",
    decay_cols
)

decay_only NDCG@10 = 0.120648


In [43]:
def add_history_features(pairs):
    result = []

    for cutoff, part in pairs.groupby("cutoff"):
        h = df_small[df_small["last_watch_dt"] < cutoff]
        f = part.copy()

        for key, name in [
            ("user_id", "user"),
            ("item_id", "item")
        ]:
            counts = h.groupby(key).size()
            means = h.groupby(key)["watched_pct"].mean()

            f[f"hist_{name}_count"] = (
                f[key].map(counts).fillna(0)
            )

            f[f"hist_{name}_mean_pct"] = (
                f[key].map(means).fillna(0)
            )

        result.append(f)

    return pd.concat(result, ignore_index=True)


train_af = add_history_features(train_af)
val_af = add_history_features(val_af)
val_fixed = add_history_features(val_fixed)

manual_extra = [
    "hist_user_count",
    "hist_user_mean_pct",
    "hist_item_count",
    "hist_item_mean_pct"
]

base_cols = features + manual_extra

In [44]:
def add_last_n_features(pairs, Ns=(3, 7, 14, 30)):
    result = []

    for cutoff, part in pairs.groupby("cutoff"):
        h = df_small[
            df_small["last_watch_dt"] < cutoff
        ][[
            "user_id", "item_id",
            "last_watch_dt", "watched_pct"
        ]].sort_values("last_watch_dt").copy()

        h["main_genre"] = (
            h["item_id"].map(genre_map).fillna("unknown")
        )

        f = part.copy()

        if "main_genre" not in f.columns:
            f["main_genre"] = (
                f["item_id"].map(genre_map).fillna("unknown")
            )

        ug_idx = pd.MultiIndex.from_frame(
            f[["user_id", "main_genre"]]
        )

        for n in Ns:
            hu = h.groupby("user_id", sort=False).tail(n)
            hi = h.groupby("item_id", sort=False).tail(n)

            # Средний процент просмотра среди последних N
            u_mean = hu.groupby("user_id")["watched_pct"].mean()
            i_mean = hi.groupby("item_id")["watched_pct"].mean()

            f[f"last{n}_user_mean_pct"] = (
                f["user_id"].map(u_mean).fillna(0).to_numpy()
            )

            f[f"last{n}_item_mean_pct"] = (
                f["item_id"].map(i_mean).fillna(0).to_numpy()
            )

            # Доля интересующего жанра среди последних N
            genre_counts = hu.groupby(
                ["user_id", "main_genre"]
            ).size()

            user_counts = hu.groupby("user_id").size()

            numerator = genre_counts.reindex(
                ug_idx, fill_value=0
            ).to_numpy()

            denominator = (
                f["user_id"].map(user_counts)
                .fillna(0).clip(lower=1).to_numpy()
            )

            f[f"last{n}_genre_share"] = (
                numerator / denominator
            )

            # Сколько дней охватывают последние N
            item_span = (
                hi.groupby("item_id")["last_watch_dt"].max()
                - hi.groupby("item_id")["last_watch_dt"].min()
            ).dt.total_seconds() / 86400

            f[f"last{n}_item_span"] = (
                f["item_id"].map(item_span).fillna(0).to_numpy()
            )

        result.append(f)

    return pd.concat(result, ignore_index=True)

In [45]:
train_af = add_last_n_features(train_af)
val_af = add_last_n_features(val_af)
val_fixed = add_last_n_features(val_fixed)

lastn_cols = [
    f"last{n}_{name}"
    for n in [3, 7, 14, 30]
    for name in [
        "user_mean_pct",
        "item_mean_pct",
        "genre_share",
        "item_span"
    ]
]

print("Last-N признаков:", len(lastn_cols))

Last-N признаков: 16


In [46]:
probe = CatBoostClassifier(
    iterations=180,
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    verbose=False
)

probe_cols = base_cols + decay_cols

probe.fit(
    train_af[probe_cols],
    train_af["label"],
    cat_features=cat_features
)

imp = pd.Series(
    probe.get_feature_importance(),
    index=probe_cols
)

adaptive_cols = []

for entity in ["item", "user_genre"]:
    candidates = [
        f"decay_{entity}_{tau}"
        for tau in [3, 14, 30]
    ]

    adaptive_cols.append(
        imp[candidates].idxmax()
    )

print("Выбранная память:", adaptive_cols)

Выбранная память: ['decay_item_14', 'decay_user_genre_14']


In [47]:
variants = {
    "Manual+": base_cols,
    "Fixed Windows": base_cols + auto_cols,
    "Last-N": base_cols + lastn_cols,
    "Exponential": base_cols + decay_cols,
    "Adaptive Memory": base_cols + adaptive_cols
}

results = []
trained_models = {}

for name, cols in variants.items():
    m = CatBoostClassifier(
        iterations=600,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        random_seed=42,
        verbose=False
    )

    m.fit(
        train_af[cols],
        train_af["label"],
        cat_features=cat_features,
        eval_set=(
            val_af[cols],
            val_af["label"]
        ),
        early_stopping_rounds=60
    )

    val_fixed["score_variant"] = (
        m.predict_proba(val_fixed[cols])[:, 1]
    )

    score = evaluate_ndcg(
        val_fixed,
        "score_variant",
        "2021-08-09"
    )

    trained_models[name] = m

    results.append({
        "method": name,
        "ndcg": score,
        "num_features": len(cols)
    })

df_results = pd.DataFrame(results)
print(df_results.sort_values("ndcg", ascending=False))

            method      ndcg  num_features
3      Exponential  0.126334            25
4  Adaptive Memory  0.125288            21
2           Last-N  0.124971            35
0          Manual+  0.124101            19
1    Fixed Windows  0.122067            38


In [48]:
cutoff = pd.Timestamp("2021-08-09")

history_counts = (
    df_small[df_small["last_watch_dt"] < cutoff]
    .groupby("user_id")
    .size()
)

val_fixed["activity_group"] = pd.cut(
    val_fixed["user_id"].map(history_counts),
    bins=[0, 5, 15, np.inf],
    labels=["1–5", "6–15", "16+"]
)

for name, cols in variants.items():
    val_fixed[f"pred_{name}"] = (
        trained_models[name]
        .predict_proba(val_fixed[cols])[:, 1]
    )

for group, sub in val_fixed.groupby("activity_group", observed=True):
    print(f"\nГруппа: {group}, пользователей: {sub['user_id'].nunique()}")

    for name in variants:
        score = evaluate_ndcg(
            sub,
            f"pred_{name}",
            "2021-08-09"
        )
        print(f"{name}: {score:.5f}")


Группа: 1–5, пользователей: 1159
Manual+: 0.22771
Fixed Windows: 0.20773
Last-N: 0.22466
Exponential: 0.21849
Adaptive Memory: 0.22981

Группа: 6–15, пользователей: 1265
Manual+: 0.13355
Fixed Windows: 0.13600
Last-N: 0.13128
Exponential: 0.13969
Adaptive Memory: 0.13446

Группа: 16+, пользователей: 576
Manual+: 0.09404
Fixed Windows: 0.09283
Last-N: 0.09805
Exponential: 0.09605
Adaptive Memory: 0.09523


In [49]:
routing = {
    "1–5": "Adaptive Memory",
    "6–15": "Exponential",
    "16+": "Last-N"
}

val_fixed["score_routed"] = np.select(
    [
        val_fixed["activity_group"].astype(str).eq(g)
        for g in routing
    ],
    [
        val_fixed[f"pred_{name}"]
        for name in routing.values()
    ],
    default=val_fixed["pred_Manual+"]
)

print(
    "Activity-Aware NDCG@10:",
    evaluate_ndcg(
        val_fixed,
        "score_routed",
        "2021-08-09"
    )
)

Activity-Aware NDCG@10: 0.1286741569482157


In [50]:
test_cutoff = pd.Timestamp("2021-08-16")

df_test_fixed = build_eval_pairs(
    df_small, test_cutoff, eval_users, k=200
)

test_fixed = add_features(df_test_fixed)
test_fixed = generate_auto_features(test_fixed)
test_fixed = add_history_features(test_fixed)
test_fixed = add_last_n_features(test_fixed)
test_fixed = add_decay_features(test_fixed)

history_counts = (
    df_small[df_small["last_watch_dt"] < test_cutoff]
    .groupby("user_id")
    .size()
)

test_fixed["activity_group"] = pd.cut(
    test_fixed["user_id"].map(history_counts),
    bins=[0, 5, 15, np.inf],
    labels=["1–5", "6–15", "16+"]
)

for name in ["Manual+", "Exponential", "Last-N", "Adaptive Memory"]:
    test_fixed[f"pred_{name}"] = (
        trained_models[name]
        .predict_proba(test_fixed[variants[name]])[:, 1]
    )

routing = {
    "1–5": "Adaptive Memory",
    "6–15": "Exponential",
    "16+": "Last-N"
}

test_fixed["score_routed"] = np.select(
    [
        test_fixed["activity_group"].astype(str).eq(group)
        for group in routing
    ],
    [
        test_fixed[f"pred_{name}"]
        for name in routing.values()
    ],
    default=test_fixed["pred_Manual+"]
)

test_fixed["pred_All AutoFE"] = (
    model_auto.predict_proba(test_fixed[all_features])[:, 1]
)

for name, col in {
    "Manual+": "pred_Manual+",
    "All AutoFE": "pred_All AutoFE",
    "Exponential": "pred_Exponential",
    "Activity-Aware": "score_routed"
}.items():
    print(
        name,
        evaluate_ndcg(test_fixed, col, test_cutoff)
    )

Manual+ 0.10464504211236916
All AutoFE 0.10428755632953834
Exponential 0.10774429543436233
Activity-Aware 0.10771000012911165


In [51]:
name = "Fixed Windows"

test_fixed["pred_Fixed_Windows"] = (
    trained_models[name].predict_proba(
        test_fixed[variants[name]]
    )[:, 1]
)

print(
    "Fixed Windows:",
    evaluate_ndcg(
        test_fixed,
        "pred_Fixed_Windows",
        "2021-08-16"
    )
)

Fixed Windows: 0.10619106348026669


In [52]:
cutoff = pd.Timestamp("2021-08-16")

history = df_small[
    df_small["last_watch_dt"] < cutoff
]

future = df_small[
    (df_small["last_watch_dt"] >= cutoff)
    & (df_small["last_watch_dt"] < cutoff + pd.Timedelta(days=7))
    & (df_small["watched_pct"] >= 50)
]

seen = history.groupby("user_id")["item_id"].agg(set).to_dict()
truth = future.groupby("user_id")["item_id"].agg(set).to_dict()

score_cols = {
    "Manual+": "pred_Manual+",
    "Exponential": "pred_Exponential",
    "Activity-Aware": "score_routed"
}

rows = []

for uid, group in test_fixed.groupby("user_id"):
    relevant = truth.get(uid, set()) - seen.get(uid, set())

    if not relevant:
        continue

    idcg = sum(
        1 / np.log2(j + 2)
        for j in range(min(len(relevant), 10))
    )

    row = {"user_id": uid}

    for name, col in score_cols.items():
        top = group.nlargest(10, col)["item_id"]

        dcg = sum(
            (item in relevant) / np.log2(j + 2)
            for j, item in enumerate(top)
        )

        row[name] = dcg / idcg

    rows.append(row)

user_metrics = pd.DataFrame(rows)

print("Пользователей в NDCG:", len(user_metrics))
print(user_metrics[list(score_cols)].mean())

Пользователей в NDCG: 254
Manual+           0.104645
Exponential       0.107744
Activity-Aware    0.107710
dtype: float64


In [53]:
rng = np.random.default_rng(42)
B = 2000
n = len(user_metrics)

indices = rng.integers(0, n, size=(B, n))

comparisons = [
    ("Exponential", "Manual+"),
    ("Activity-Aware", "Exponential")
]

for a, b in comparisons:
    diffs = (
        user_metrics[a] - user_metrics[b]
    ).to_numpy()

    bootstrap = diffs[indices].mean(axis=1)

    ci = np.quantile(bootstrap, [0.025, 0.975])

    print(
        f"{a} - {b}: "
        f"delta={diffs.mean():.6f}, "
        f"95% CI=[{ci[0]:.6f}, {ci[1]:.6f}]"
    )

Exponential - Manual+: delta=0.003099, 95% CI=[-0.005649, 0.012274]
Activity-Aware - Exponential: delta=-0.000034, 95% CI=[-0.004267, 0.004554]


In [54]:
from time import perf_counter

raw = df_val_fixed

t0 = perf_counter()
manual_bench = add_features(raw)
t_manual = perf_counter() - t0

t0 = perf_counter()
base_bench = add_history_features(manual_bench)
t_history = perf_counter() - t0

t0 = perf_counter()
fixed_bench = generate_auto_features(base_bench)
t_fixed = perf_counter() - t0

t0 = perf_counter()
exp_bench = add_decay_features(base_bench)
t_exp = perf_counter() - t0

print("Кандидатных пар:", len(raw))
print(f"Manual features: {t_manual:.2f} s")
print(f"History features: {t_history:.2f} s")
print(f"Fixed Windows: {t_fixed:.2f} s")
print(f"Exponential: {t_exp:.2f} s")

for name, data in [
    ("Manual+", base_bench),
    ("Fixed Windows", fixed_bench),
    ("Exponential", exp_bench)
]:
    cols = variants[name]
    model = trained_models[name]

    t0 = perf_counter()
    _ = model.predict_proba(data[cols])[:, 1]
    inference = perf_counter() - t0

    memory = (
        data[cols].memory_usage(deep=True).sum()
        / 1024**2
    )

    print(
        f"{name}: "
        f"features={len(cols)}, "
        f"inference={inference:.3f}s, "
        f"feature_memory={memory:.1f}MB"
    )

Кандидатных пар: 600000
Manual features: 1.06 s
History features: 0.16 s
Fixed Windows: 0.73 s
Exponential: 0.51 s
Manual+: features=19, inference=1.735s, feature_memory=110.4MB
Fixed Windows: features=38, inference=1.517s, feature_memory=153.9MB
Exponential: features=25, inference=1.573s, feature_memory=137.8MB


In [55]:
import sys
import numpy as np
import pandas as pd
import catboost
import sklearn

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("CatBoost:", catboost.__version__)
print("scikit-learn:", sklearn.__version__)

for name, model in trained_models.items():
    print(
        name,
        "trees:", model.tree_count_,
        "features:", len(variants[name])
    )

Python: 3.13.12
NumPy: 2.5.2
pandas: 3.0.5
CatBoost: 1.2.10
scikit-learn: 1.9.0
Manual+ trees: 229 features: 19
Fixed Windows trees: 137 features: 38
Last-N trees: 138 features: 35
Exponential trees: 228 features: 25
Adaptive Memory trees: 215 features: 21
